# E1 UNet Baseline on Preprocessed BTXRD 224x224

Kaggle notebook for training the clean image-only UNet baseline on GPU T4. Attach a Kaggle Dataset that contains the exported BTXRD structure:

```text
data/exports/btxrd_preprocessed/train.csv
data/exports/btxrd_preprocessed/val.csv
data/exports/btxrd_preprocessed/test.csv
data/processed/images_preprocessed/
data/processed/masks_preprocessed/
```

If your Kaggle Dataset root is different, edit only `DATA_ROOT` in the setup cell.

In [ ]:
import os
import sys
import json
import shutil
import zipfile
from pathlib import Path

import torch

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Setup Paths

The notebook expects the repo to be available in `/kaggle/working/BTXRD-LViT`. If you uploaded this notebook inside the repo, the auto-detection below usually works. If not, clone your repo or upload the repo files into Kaggle working storage before continuing.

In [ ]:
# Edit these two paths if your Kaggle layout differs.
REPO_ROOT = Path('/kaggle/working/BTXRD-LViT')
DATA_ROOT = None

def find_repo_root():
    candidates = [
        Path.cwd(),
        Path('/kaggle/working/BTXRD-LViT'),
        Path('/kaggle/input/BTXRD-LViT'),
    ]
    for candidate in candidates:
        if (candidate / 'src' / 'data' / 'btxrd_dataset.py').exists():
            return candidate
    return REPO_ROOT

def find_data_root():
    expected = Path('data/exports/btxrd_preprocessed/train.csv')
    candidates = [Path.cwd(), Path('/kaggle/working/BTXRD-LViT')]
    candidates.extend(sorted(Path('/kaggle/input').glob('*')) if Path('/kaggle/input').exists() else [])
    for candidate in candidates:
        if (candidate / expected).exists():
            return candidate
    raise FileNotFoundError('Could not auto-detect DATA_ROOT. Set DATA_ROOT manually to the dataset root.')

REPO_ROOT = find_repo_root()
DATA_ROOT = find_data_root() if DATA_ROOT is None else Path(DATA_ROOT)
OUTPUT_DIR = Path('/kaggle/working/experiments/E1_unet_preprocessed_224_full_labels')
RUNTIME_CONFIG = Path('/kaggle/working/e1_unet_kaggle.yaml')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(REPO_ROOT))
print('REPO_ROOT:', REPO_ROOT)
print('DATA_ROOT:', DATA_ROOT)
print('OUTPUT_DIR:', OUTPUT_DIR)

In [ ]:
required_paths = [
    DATA_ROOT / 'data/exports/btxrd_preprocessed/train.csv',
    DATA_ROOT / 'data/exports/btxrd_preprocessed/val.csv',
    DATA_ROOT / 'data/exports/btxrd_preprocessed/test.csv',
    DATA_ROOT / 'data/processed/images_preprocessed',
    DATA_ROOT / 'data/processed/masks_preprocessed',
]

missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required BTXRD paths:\n' + '\n'.join(missing))

print('BTXRD dataset paths are ready.')

## Runtime Config

The source config stays unchanged. This cell writes a Kaggle-specific copy with `root_dir`, `num_workers`, and output path adjusted for `/kaggle/working`.

In [ ]:
import yaml

BASE_CONFIG = REPO_ROOT / 'configs/train_unet_baseline.yaml'
with BASE_CONFIG.open('r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

cfg['data']['root_dir'] = str(DATA_ROOT)
cfg['training']['device'] = 'cuda'
cfg['training']['image_size'] = 224
cfg['training']['batch_size'] = 4
cfg['training']['num_workers'] = 2
cfg['training']['epochs'] = 200
cfg['training']['learning_rate'] = 0.0003
cfg['training']['early_stopping_patience'] = 100
cfg['training']['output_dir'] = str(OUTPUT_DIR)

with RUNTIME_CONFIG.open('w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print(RUNTIME_CONFIG.read_text())

## Smoke Test

Run this before the long training job. It checks dataset loading, binary masks, UNet forward pass, and backward pass.

In [ ]:
!python {REPO_ROOT}/src/training/smoke_test_model_pipeline.py --config {RUNTIME_CONFIG} --samples-per-split 8

## Train E1 UNet Baseline

This is the main run. On Kaggle T4, keep GPU enabled: Notebook Settings -> Accelerator -> GPU T4.

In [ ]:
!python {REPO_ROOT}/src/training/train_unet.py --config {RUNTIME_CONFIG} --device cuda

## Evaluate Best Checkpoint

Metrics are written separately for validation and test. For normal cases, prioritize `normal_pred_area_ratio` and `normal_fp_image_rate` in the report.

In [ ]:
BEST_CKPT = OUTPUT_DIR / 'best.pt'
assert BEST_CKPT.exists(), f'Missing checkpoint: {BEST_CKPT}'

!python {REPO_ROOT}/src/training/evaluate_unet.py --config {RUNTIME_CONFIG} --checkpoint {BEST_CKPT} --split val --device cuda --output {OUTPUT_DIR}/val_metrics.json
!python {REPO_ROOT}/src/training/evaluate_unet.py --config {RUNTIME_CONFIG} --checkpoint {BEST_CKPT} --split test --device cuda --output {OUTPUT_DIR}/test_metrics.json

In [ ]:
def load_metrics(path):
    with Path(path).open('r', encoding='utf-8') as f:
        return json.load(f)

for split in ['val', 'test']:
    metrics = load_metrics(OUTPUT_DIR / f'{split}_metrics.json')
    print(f'[{split}]')
    for key in ['tumor_dice', 'tumor_iou', 'normal_pred_area_ratio', 'normal_fp_image_rate']:
        print(f'  {key}: {metrics[key]:.6f}')

## Package Artifacts

The zip file appears under `/kaggle/working` and can be downloaded from the Kaggle output panel.

In [ ]:
ARTIFACT_ZIP = Path('/kaggle/working/E1_unet_preprocessed_224_full_labels_artifacts.zip')
artifact_names = [
    'best.pt',
    'last.pt',
    'history.csv',
    'best_summary.json',
    'val_metrics.json',
    'test_metrics.json',
    'config.json',
]

with zipfile.ZipFile(ARTIFACT_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for name in artifact_names:
        path = OUTPUT_DIR / name
        if path.exists():
            zf.write(path, arcname=name)
        else:
            print('Skipping missing artifact:', path)

print('Wrote:', ARTIFACT_ZIP)
print('Size MB:', ARTIFACT_ZIP.stat().st_size / (1024 * 1024))